# Day 4 — Transformers, Tokens, Tokenization & LLM Context

**AI Core Track — Personal Learning Notebook**

This notebook combines:

- Day 4 course theory
- Hands-on practice from the Day 4 practice notebook
- Working Python examples
- OpenAI and Gemini API examples
- Personal notes and corrections from my learning

## Day 4 Topics

1. Transformers — the architecture behind modern GPT-style LLMs
2. From LSTMs to Transformers
3. Attention and why it matters
4. Parameters — millions to trillions
5. Tokens and tokenizers
6. Tokenization with `tiktoken`
7. Tokenization with Hugging Face
8. The illusion of LLM memory / stateless requests
9. Context windows, token limits and API cost
10. OpenAI current Python API
11. Gemini current Python API
12. Practical lessons and common mistakes


## 1. Transformers — The Architecture Behind Modern LLMs

The **Transformer** is the architecture behind many modern language models.

The key idea is **attention**: instead of processing a sequence strictly one step at a time, the model can learn relationships between tokens in the context.

A simplified view:

```text
Input text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Embeddings + positional information
   ↓
Transformer blocks
   ├── Self-Attention
   ├── Feed-Forward Network
   ├── Normalization
   └── Residual connections
   ↓
Output probabilities
   ↓
Next token prediction
```

### Why Transformers became important

Traditional recurrent architectures such as RNNs/LSTMs process sequences sequentially. Transformers use attention to model relationships between tokens more directly and can be trained efficiently with substantial parallelism.

Modern LLMs are not simply "a Transformer"; they are large systems built using Transformer-based architectures plus training, data, inference, safety, tool use, and other engineering components.


## 2. From LSTMs to Transformers

### RNN / LSTM idea

An LSTM maintains a hidden state while processing a sequence:

```text
Token 1 → LSTM → State
                 ↓
Token 2 → LSTM → State
                 ↓
Token 3 → LSTM → State
```

This works well for sequential data, but long-range dependencies can be difficult to represent efficiently.

### Transformer idea

Self-attention allows each token to consider other relevant tokens in the context:

```text
"The animal didn't cross the road because it was tired."

                         ↑
                  "it" can attend
                  to relevant context
```

The attention mechanism learns which parts of the context are useful for the current representation.


## 3. Attention

A simplified attention idea is:

```text
Query (Q)   → What am I looking for?
Key (K)     → What information does each token contain?
Value (V)   → What information should be retrieved?
```

The standard scaled dot-product attention is commonly expressed as:

```text
Attention(Q, K, V)
    = softmax(QKᵀ / √dₖ)V
```

### Intuition

1. Compare a query with keys.
2. Produce attention scores.
3. Normalize the scores with softmax.
4. Use those scores to combine the values.

### Multi-head attention

Instead of one attention operation, Transformer blocks can use multiple attention heads so the model can learn different relationships in parallel.


## 4. Parameters — Millions to Billions and Beyond

A **parameter** is a learned numerical value inside a neural network.

During training, parameters are adjusted so the model becomes better at its task.

Conceptually:

```text
Training data
     ↓
Model
     ↓
Loss
     ↓
Backpropagation + optimization
     ↓
Updated parameters
```

More parameters generally mean a larger model, but **parameter count alone does not determine model quality**.

Other factors matter:

- Training data
- Data quality
- Architecture
- Training compute
- Optimization
- Post-training
- Inference-time reasoning
- Tool use
- Context handling

### Important distinction

```text
Parameters ≠ Tokens ≠ Embeddings

Parameters → learned model weights
Tokens     → pieces of input/output text
Embeddings → numerical vector representations
```


## 5. What Are Tokens?

A **token** is a piece of text processed by a language model's tokenizer.

A token can be:

- A whole word
- Part of a word
- Punctuation
- Whitespace-associated text
- A special token

For example, a tokenizer might split:

```text
"unbelievable"
        ↓
["un", "believ", "able"]
```

The exact split depends on the tokenizer.

### Token IDs

After tokenization, tokens are mapped to vocabulary IDs:

```text
Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Token IDs
 ↓
LLM
```

Token IDs are **not embeddings**.

```text
Token ID:
[15496, 995, ...]

Embedding:
[0.021, -0.183, 0.492, ...]
```

Token IDs identify vocabulary items. Embeddings are dense numerical representations used for semantic/model computations.


In [ ]:
# Simple tokenization concept with Hugging Face

%pip install -q transformers


In [ ]:
from transformers import AutoTokenizer

# Correct public model identifier:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hi how are you?"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Number of token IDs:", len(token_ids))


### A lesson from my Day 4 practice

I initially used:

```text
bert-base-uncase
```

The correct identifier is:

```text
bert-base-uncased
```

A model-name typo can produce a confusing Hugging Face error that mentions `401`, repository-not-found, or authentication.

For public models, first verify the exact model ID before assuming you need a Hugging Face token.


## 6. Tokenizing with `tiktoken`

`tiktoken` is a tokenizer library commonly used with OpenAI model families.

For a model-specific tokenizer:

```python
encoding = tiktoken.encoding_for_model("model-name")
```

Then:

```python
tokens = encoding.encode(text)
```

The result is a list of token IDs.

To decode:

```python
encoding.decode(tokens)
```

To inspect each token:

```python
for token_id in tokens:
    print(token_id, encoding.decode([token_id]))
```

The exact tokenization depends on the model/tokenizer.


In [ ]:
%pip install -q tiktoken


In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

text = "Hi my name is Ed and I like banoffee pie"

tokens = encoding.encode(text)

print("Token IDs:", tokens)
print("Number of tokens:", len(tokens))
print("Decoded:", encoding.decode(tokens))


In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text!r}")


### Important correction from practice

Do not use:

```python
encoding = tiktoken.Auto()
```

That is not the normal `tiktoken` API.

Use:

```python
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
```

when you want a model-associated encoding.


## 7. The Illusion of "Memory"

A basic API request is best understood as a request/response operation.

If you send:

```text
User: Hi! I'm Ed!
```

and later send only:

```text
User: What's my name?
```

the second request does not automatically contain the first message just because the model saw it earlier.

### Stateless request model

```text
Request 1
User: Hi! I'm Ed!
        ↓
      Model
        ↓
    Response 1


Request 2
User: What's my name?
        ↓
      Model
        ↓
    Response 2
```

If the second request needs the first message, your application can provide the conversation history again:

```text
System instruction
+
Previous user message
+
Previous assistant response
+
New user message
        ↓
       LLM
```

This creates the **illusion of conversational memory**.

### Important nuance

Some modern APIs and products provide conversation/state-management features. That does not change the core engineering idea: the model needs access to the relevant context/state to use previous information.


In [ ]:
# Conversation history as application-managed context

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I help you?"},
    {"role": "user", "content": "What's my name?"}
]

messages


## 8. Context Windows, Token Limits & Cost

A model's **context window** is the amount of tokenized information it can process for a request/conversation according to that model/API.

The context can include:

- System/developer instructions
- User messages
- Previous assistant messages
- Retrieved documents
- Tool results
- Images/files or other supported inputs
- The current request

### Why token count matters

Token usage affects:

- Context-window limits
- Input/output limits
- API usage and cost
- Latency
- How much history or retrieved information can fit

A useful mental model:

```text
Context
= instructions
+ conversation
+ retrieved information
+ tool results
+ current request
```

As the context grows, an application may need techniques such as summarization, retrieval, context selection, or caching.


## 9. OpenAI — Current Python API

For **new OpenAI applications**, the current recommended interface is the **Responses API**.

Official current Python pattern:

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.6",
    input="Explain what a Transformer is in one sentence."
)

print(response.output_text)
```

The OpenAI SDK can read `OPENAI_API_KEY` from the environment.

For a `.env` file in local development:

```text
OPENAI_API_KEY=your_key_here
```

Then:

```python
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()
```

### Why not use Chat Completions here?

Chat Completions is still a real API interface, but for new OpenAI work you should learn the **Responses API first** and check the current official documentation before copying older tutorials.


In [ ]:
# OpenAI current Python example

%pip install -q openai python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-5.6",
    input="Explain what a Transformer is in one sentence."
)

print(response.output_text)


### OpenAI conversation-style input

The Responses API also accepts structured input items. For example:

```python
response = client.responses.create(
    model="gpt-5.6",
    input=[
        {
            "role": "developer",
            "content": "You are a concise tutor."
        },
        {
            "role": "user",
            "content": "What is attention in Transformers?"
        }
    ]
)
```

This is useful when your application needs explicit roles and multiple turns/items.


In [ ]:
# Structured input example

response = client.responses.create(
    model="gpt-5.6",
    input=[
        {
            "role": "developer",
            "content": "You are a concise AI engineering tutor."
        },
        {
            "role": "user",
            "content": "What is self-attention?"
        }
    ]
)

print(response.output_text)


## 10. Gemini — Current Python API

Google's current Python SDK is:

```text
google-genai
```

Google's current documentation provides the **Interactions API** as the newer interface, while `models.generate_content()` is documented as the previous/legacy interface.

### Current Interactions pattern

```python
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain what a Transformer is in one sentence."
)

print(interaction.output_text)
```

The SDK can use `GEMINI_API_KEY` from the environment.

`.env`:

```text
GEMINI_API_KEY=your_key_here
```

Then:

```python
from dotenv import load_dotenv
load_dotenv()

client = genai.Client()
```


In [ ]:
# Gemini current Python SDK

%pip install -q -U google-genai python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain what a Transformer is in one sentence."
)

print(interaction.output_text)


### Gemini `generate_content()` — still useful to know

The Google documentation also documents:

```python
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what a Transformer is."
)

print(response.text)
```

This is useful because existing code, tutorials, and projects may use it.

The important thing is **not to mix the interfaces**:

```text
Interactions API
input → interaction.output_text

Generate Content
contents → response.text
```

Google's current documentation recommends the newer Interactions API for access to the latest features and models.


In [ ]:
# Gemini Generate Content interface

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what a Transformer is in one sentence."
)

print(response.text)


## 11. OpenAI vs Gemini — Side-by-Side

### Native Python SDKs

```text
OpenAI
  SDK: openai
  Client: OpenAI()
  Current API: Responses
  Input: input
  Text output: response.output_text

Gemini
  SDK: google-genai
  Client: genai.Client()
  Current interface: Interactions
  Input: input
  Text output: interaction.output_text
```

### Another Gemini interface

```text
Gemini Generate Content
  client.models.generate_content()
  contents=...
  response.text
```

### Key lesson

The names are similar but **API interfaces are provider-specific**.

Do not assume:

```python
input=...
```

works everywhere, or that every response has:

```python
response.text
```

Always check the provider's current API reference.


## 12. Tokenization vs Embeddings

These concepts are easy to confuse.

### Tokenization

```text
Text
 ↓
Tokenizer
 ↓
Token IDs
```

Example:

```text
"Hello world"
      ↓
[15496, 995]
```

### Embeddings

```text
Text
 ↓
Embedding Model
 ↓
Dense vector
```

Example:

```text
[0.021, -0.183, 0.492, ...]
```

An embedding is not simply a longer list of token IDs.

Embeddings are typically used for:

- Semantic search
- Retrieval
- RAG
- Similarity
- Clustering
- Recommendations


## 13. Parameters vs Tokens vs Context

### Parameters

Learned values inside the model.

```text
Model weights → parameters
```

### Tokens

Pieces of text processed by the tokenizer.

```text
Text → tokens → token IDs
```

### Context

The information supplied to the model for a particular request.

```text
Context
= instructions
+ messages
+ retrieved data
+ tool outputs
+ current input
```

### Embeddings

Numerical vectors representing information in an embedding space.

```text
Text → embedding vector
```

Remember:

```text
Parameters → belong to the model
Tokens     → represent text pieces
Context    → information supplied to a request
Embeddings → vectors for representation/similarity
```


## 14. Practical API Architecture

The concepts from this day fit together like this:

```text
                    Your Application
                           │
                    Client Library / SDK
                    ┌──────┴──────┐
                    ↓             ↓
                 OpenAI         Gemini
                    ↓             ↓
               Responses      Interactions
                    ↓             ↓
                 Model          Model
                    │             │
                    └──────┬──────┘
                           ↓
                        Output
```

For local models:

```text
Your Application
       ↓
Ollama / vLLM / Transformers
       ↓
Local or self-hosted model
```

The SDK is the communication layer. The model is the learned system. The endpoint/API is the service interface.


## 15. Common Errors & Lessons From My Practice

### 1. Wrong Hugging Face model ID

Wrong:

```text
bert-base-uncase
```

Correct:

```text
bert-base-uncased
```

### 2. Mixing Gemini API interfaces

Wrong concept:

```python
client.models.generate_content(
    model="...",
    input="..."
)
```

Use:

```python
client.models.generate_content(
    model="...",
    contents="..."
)
```

Or use the Interactions API:

```python
client.interactions.create(
    model="...",
    input="..."
)
```

### 3. Mixing response properties

Generate Content:

```python
response.text
```

Interactions:

```python
interaction.output_text
```

OpenAI Responses:

```python
response.output_text
```

### 4. API key problems

Keep keys in `.env` and never commit the real `.env` file to GitHub.

```text
OPENAI_API_KEY=...
GEMINI_API_KEY=...
```

### 5. Tokenizer is model-dependent

Different model families can use different tokenizers. Don't assume token IDs from one tokenizer are valid for another model.


## 16. Quick Revision Sheet

### Transformers

> Architecture based heavily on attention and used by many modern LLMs.

### Attention

> Mechanism that lets representations weigh relevant parts of the context.

### Parameters

> Learned numerical values of a model.

### Token

> A piece of text produced by a tokenizer.

### Token ID

> Numerical ID representing a token in a tokenizer vocabulary.

### Context Window

> The amount of tokenized information a model/API can handle for a request according to its limits.

### Statelessness

> A basic API request does not automatically remember previous requests; the application/API must provide or manage the required context/state.

### Tokenization

```text
Text → Tokenizer → Token IDs
```

### Embedding

```text
Text → Embedding Model → Vector
```

### Current API patterns learned

```text
OpenAI
client.responses.create(...)
response.output_text

Gemini
client.interactions.create(...)
interaction.output_text
```

### Legacy/previous Gemini interface to recognize

```text
client.models.generate_content(...)
response.text
```


## 17. Official Documentation to Check Before Coding

APIs and model names change frequently.

Before copying a snippet from an old tutorial, verify:

```text
Provider
   ↓
Current model
   ↓
Current API
   ↓
Current Python SDK
   ↓
Current authentication
   ↓
Current pricing / limits
```

Useful official documentation:

- OpenAI API documentation: https://platform.openai.com/docs
- Google Gemini API documentation: https://ai.google.dev/gemini-api/docs
- Hugging Face Transformers documentation: https://huggingface.co/docs/transformers
- tiktoken repository/documentation: https://github.com/openai/tiktoken

> **Official documentation is the source of truth.**


## 18. My Day 4 Takeaways

Today I moved from simply calling an LLM API to understanding some of the mechanisms underneath LLM applications:

- Transformers provide the core architecture behind many modern LLMs.
- Attention is central to Transformer-based language modeling.
- Parameters are learned weights; they are different from tokens.
- Tokenization converts text into token IDs.
- `tiktoken` lets me inspect tokenization used by supported OpenAI model families.
- Hugging Face tokenizers let me inspect tokenization for open-weight model families.
- LLM API calls should be treated carefully with respect to context and state.
- What looks like "memory" can come from sending previous conversation context.
- Context windows and token counts affect what an application can send and what it costs.
- OpenAI and Gemini have different native Python SDKs and API interfaces.
- I should verify current official documentation before using an API snippet.


## 19. Practice Tasks

### Task 1
Tokenize your own sentence with:

- `tiktoken`
- Hugging Face `AutoTokenizer`

Compare the token counts.

### Task 2
Create a conversation with 3 user/assistant turns and inspect the message history.

### Task 3
Call the same prompt using:

- OpenAI Responses API
- Gemini Interactions API

Compare the response format.

### Task 4
Increase the amount of conversation context and observe how the request grows.

### Task 5
Explain from memory:

```text
Token
Token ID
Embedding
Parameter
Context Window
API Endpoint
SDK
```

---

**Day 4 completed — Transformers, attention, parameters, tokens, tokenization, context and API practice.**
